# 🩸 Pima Indians Diabetes — Model Training Pipeline
### Minor Project Part-I: Metabolic Classifier Architecture & Fitting
**Dataset:** Pima Indians Diabetes (`processed_data/cleaned_diabetes.csv`)  
**Scope:** **Task 3 — Training Only (No Evaluation)**  
> **Note:** As specified in Task 3, this notebook strictly focuses on **data partitioning, feature scaling, model instantiation, fitting on training data, and model serialization**. No evaluation metrics, test scoring, or confusion matrices are performed in this phase.

## 1. Setup & Library Imports
Import scikit-learn estimators and persistence tools.

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

# Scikit-learn estimators & preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

BASE_DIR = os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, "processed_data", "cleaned_diabetes.csv")
MODEL_DIR = os.path.join(BASE_DIR, "saved_models")
os.makedirs(MODEL_DIR, exist_ok=True)

print("Environment initialized successfully.")
print(f"Target Directory for Saved Models: {MODEL_DIR}")

Environment initialized successfully.
Target Directory for Saved Models: C:\Users\Lakshya\OneDrive\Desktop\Minor-Project-1\saved_models


## 2. Data Loading & Partitioning
Load cleaned diabetes dataset (with imputed physiological zeros) and perform an 80/20 stratified train-test split ($N_{\text{train}} = 614$, $N_{\text{test}} = 154$).

In [2]:
# 2.1 Load Cleaned Data
df = pd.read_csv(DATA_PATH)
print(f"Loaded Cleaned Diabetes Data: {df.shape[0]} rows, {df.shape[1]} columns")

X = df.drop(columns=['Outcome'])
y = df['Outcome']

# 2.2 Stratified 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Features Shape (X_train) : {X_train.shape}")
print(f"Training Labels Shape (y_train)   : {y_train.shape}")
print(f"Testing Features Shape (X_test)   : {X_test.shape} (Preserved for Task 4)")
print(f"Testing Labels Shape (y_test)     : {y_test.shape}")
print("\nTarget distribution in Training Set:")
print(y_train.value_counts().rename({0: 'Non-Diabetic (0)', 1: 'Diabetic (1)'}))

Loaded Cleaned Diabetes Data: 768 rows, 9 columns
Training Features Shape (X_train) : (614, 8)
Training Labels Shape (y_train)   : (614,)
Testing Features Shape (X_test)   : (154, 8) (Preserved for Task 4)
Testing Labels Shape (y_test)     : (154,)

Target distribution in Training Set:
Outcome
Non-Diabetic (0)    400
Diabetic (1)        214
Name: count, dtype: int64


## 3. Feature Standardization
Standardize metabolic indicators using `StandardScaler` fitted strictly on `X_train`.

In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Persist scaler
scaler_path = os.path.join(MODEL_DIR, "diabetes_scaler.pkl")
joblib.dump(scaler, scaler_path)

print(f"[SAVED] Diabetes StandardScaler exported to: {scaler_path}")
print(f"Features: {X.columns.tolist()}")

[SAVED] Diabetes StandardScaler exported to: C:\Users\Lakshya\OneDrive\Desktop\Minor-Project-1\saved_models\diabetes_scaler.pkl
Features: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']


## 4. Model Instantiation & Training
Train **6 machine learning classification models** on the metabolic feature space:
1. **Logistic Regression**
2. **Decision Tree (CART)**
3. **Random Forest Classifier**
4. **Support Vector Machine (RBF Kernel)**
5. **K-Nearest Neighbors (KNN)**
6. **Gradient Boosting Classifier**

In [4]:
# 4.1 Define Models with Calibrated Hyperparameters
models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'Support Vector Machine (RBF)': SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, random_state=42)
}

trained_models = {}

print("=" * 75)
print(" EXECUTING MODEL TRAINING ON DIABETES COHORT (N = 614)")
print("=" * 75)

# 4.2 Model Fitting
for name, model in models.items():
    print(f"\nFitting [{name}] ...")
    if name in ['Logistic Regression', 'Support Vector Machine (RBF)', 'K-Nearest Neighbors']:
        model.fit(X_train_scaled, y_train)
        input_type = "Scaled Features"
    else:
        model.fit(X_train, y_train)
        input_type = "Raw Features"
        
    trained_models[name] = model
    print(f"  ✓ Training Complete. Fitted on: {input_type}")
    print(f"  ✓ Classes: {model.classes_}")

print("\n" + "=" * 75)
print(" ALL 6 DIABETES MODELS SUCCESSFULLY TRAINED!")
print("=" * 75)

 EXECUTING MODEL TRAINING ON DIABETES COHORT (N = 614)

Fitting [Logistic Regression] ...
  ✓ Training Complete. Fitted on: Scaled Features
  ✓ Classes: [0 1]

Fitting [Decision Tree] ...
  ✓ Training Complete. Fitted on: Raw Features
  ✓ Classes: [0 1]

Fitting [Random Forest] ...
  ✓ Training Complete. Fitted on: Raw Features
  ✓ Classes: [0 1]

Fitting [Support Vector Machine (RBF)] ...
  ✓ Training Complete. Fitted on: Scaled Features
  ✓ Classes: [0 1]

Fitting [K-Nearest Neighbors] ...
  ✓ Training Complete. Fitted on: Scaled Features
  ✓ Classes: [0 1]

Fitting [Gradient Boosting] ...


  ✓ Training Complete. Fitted on: Raw Features
  ✓ Classes: [0 1]

 ALL 6 DIABETES MODELS SUCCESSFULLY TRAINED!


## 5. Model Serialization & Persistence
Export trained model objects to `.pkl` binaries in `saved_models/`.

In [5]:
file_mapping = {
    'Logistic Regression': 'diabetes_logistic_regression.pkl',
    'Decision Tree': 'diabetes_decision_tree.pkl',
    'Random Forest': 'diabetes_random_forest.pkl',
    'Support Vector Machine (RBF)': 'diabetes_rbf_svm.pkl',
    'K-Nearest Neighbors': 'diabetes_knn.pkl',
    'Gradient Boosting': 'diabetes_gradient_boosting.pkl'
}

print("--- Serializing Trained Diabetes Models to Disk ---")
saved_catalog = []

for name, model in trained_models.items():
    filename = file_mapping[name]
    save_path = os.path.join(MODEL_DIR, filename)
    joblib.dump(model, save_path)
    file_size_kb = os.path.getsize(save_path) / 1024
    
    saved_catalog.append({
        'Model Name': name,
        'File Name': filename,
        'File Size (KB)': f"{file_size_kb:.2f} KB",
        'Training Status': 'Trained & Serialized'
    })

catalog_df = pd.DataFrame(saved_catalog)
display(catalog_df)
print("\nAll models and scalers are persisted and ready for evaluation in Task 4.")

--- Serializing Trained Diabetes Models to Disk ---


,Model Name,File Name,File Size (KB),Training Status
0,Logistic Regression,diabetes_logistic_regression.pkl,0.91 KB,Trained & Serialized
1,Decision Tree,diabetes_decision_tree.pkl,3.71 KB,Trained & Serialized
2,Random Forest,diabetes_random_forest.pkl,422.04 KB,Trained & Serialized
3,Support Vector Machine (RBF),diabetes_rbf_svm.pkl,27.07 KB,Trained & Serialized
4,K-Nearest Neighbors,diabetes_knn.pkl,92.58 KB,Trained & Serialized
5,Gradient Boosting,diabetes_gradient_boosting.pkl,137.91 KB,Trained & Serialized



All models and scalers are persisted and ready for evaluation in Task 4.
